Iterate through each file, extract the APK name and version, and finally store the data as an APK record.

I want to write code (in Jupyter) to iterate through all files in a specific directory and get the filename (to be used as `apk_name`). Then, I want to iterate through the files again; if a file exists with a name that closely matches the `apk_name` and has an `.apk` extension, I want to extract the numeric part of that filename to serve as the `version`. Finally, I want to output a table with columns for `apk_name` and `version`.

In [1]:
import os
import re
import pandas as pd

d:\softwall_install\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [ ]:
def extract_apk_info(root_dir):
    data = []
    # Traverse the root directory and all its subdirectories.
    for root, dirs, files in os.walk(root_dir):
        # Use the current folder name as apk_name (for example, ai.wizely.android)
        apk_name = os.path.basename(root)

        # Ignore the root directory itself and only process package directories.
        if not apk_name or apk_name == os.path.basename(root_dir):
            continue

        for file in files:
            # Check whether it is an APK file and whether the filename closely matches apk_name.
            if file.endswith('.apk') and apk_name in file:
                # Use a regular expression to extract the numeric sequence from the filename as the version.
                # For example, extract "546" from "ai.wizely.android-546.apk"
                version_match = re.search(r'-(\d+)\.apk$', file)
                if version_match:
                    version = version_match.group(1)
                    data.append({'apk_name': apk_name, 'version': version})

    # Convert to a DataFrame and drop duplicates (to avoid duplicate entries from config files in the same package folder)
    df = pd.DataFrame(data).drop_duplicates()
    return df

In [ ]:
# Set your directory path
target_path = './drive-download-20260228T163116Z-1-002'  # Update this to match your actual path
result_table = extract_apk_info(target_path)

# Display the table in Jupyter
result_table.head()

,apk_name,version
0,ai.wizely.android,546
1,app.supercube.mtlkrtn,10107
2,blockpuzzle.wood.sudoku.puzzlegames,4140
3,br.com.mobileasy.comprasparaguai,107
4,cast.video.tool.screenmirroring.casttotv,176


In [5]:
result_table.shape

(28, 2)

Write code in Jupyter that iterates through all files in a directory, gets the filename as `apk_name`, and then traverses the same directory again. If there is a file whose name closely matches the `apk_name` and ends with `.apk`, extract the numeric portion of that filename as the `version`. Finally, output a table with the columns `apk_name` and `version`.

To make sure every folder (each `apk_name`) appears in the final table even when no matching APK file is found, we need to adjust the logic: first determine the folder name, then search within that folder for a version number. If no valid APK is found, assign `version` as `None` (Null).

In [2]:
import os
import re
import pandas as pd

In [ ]:
def extract_apk_info(root_dir):
    data = []

    # 1. Get all subfolder names under the target directory (used as apk_name)
    # Assume the directory structure is something like 100APPS/drive-download.../ai.wizely.android
    for root, dirs, files in os.walk(root_dir):
        # Exclude the top-level root directory and only handle concrete package folders.
        # We locate the package folder by checking whether there are files inside it.
        if not files:
            continue

        apk_name = os.path.basename(root)
        found_version = None

        # 2. Search the current folder for the APK file and extract the version number.
        for file in files:
            if file.endswith('.apk') and apk_name in file:
                # Match the number after the hyphen, such as "546" in "-546.apk"
                match = re.search(r'-(\d+)\.apk$', file)
                if match:
                    found_version = match.group(1)

        # 3. Record the apk_name even if no matching version is found.
        data.append({
            'apk_name': apk_name,
            'version': found_version,
            'ppurl1': None,
            'wayback1': None,
        })

    # Convert to DataFrame
    df = pd.DataFrame(data)

    return df

In [ ]:
# Set your directory path (update as needed)
target_path = '1122apk/raw/drive-download-20260305T173717Z-1-002'
result_table = extract_apk_info(target_path)

# Show the result in Jupyter
# Use fillna('Null') if you want missing values to be displayed more clearly in the table
# result_table.sort_values('apk_name')

In [24]:
result_table.head()

,apk_name,version,ppurl1,wayback1
0,ae.brandsforless.android,412,None,None
1,ae.brandsforless.android,413,None,None
2,air.bg.lan.Monopoli,7000009,None,None
3,air.bg.lan.Monopoli,7000011,None,None
4,air.com.bigwigmedia.hotdogbush,2001179,None,None


In [25]:
result_table.shape

(92, 4)

In [26]:
path = target_path+"/summary.csv"
path

'1122apk/raw/drive-download-20260305T173717Z-1-002/summary.csv'

In [27]:
result_table.to_csv(path)